Load and inspect CSV

In [51]:
import pandas as pd
from pathlib import Path

RAW = Path("data/raw")
CLEAN = Path("data/clean")

CLEAN.mkdir(parents=True, exist_ok=True)

In [52]:
collisions = pd.read_csv(
    RAW / "dft-road-casualty-statistics-collision-last-5-years.csv",
    low_memory=False
)

In [53]:
# first 5 rows
collisions.head()

,collision_index,collision_year,collision_ref_no,location_easting_osgr,location_northing_osgr,longitude,latitude,police_force,collision_severity,number_of_vehicles,...,carriageway_hazards_historic,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight
0,202417H105824,2024,17H105824,448628.0,533968.0,-1.24700,54.69840,17,2,2,...,-1,0,1,1,2,E01012005,7,1,1.0,0.0
1,202517M115025,2025,17M115025,452379.0,519038.0,-1.19147,54.56387,17,2,2,...,-1,0,1,1,2,E01012090,6,1,1.0,0.0
2,202517L306225,2025,17L306225,459273.0,523811.0,-1.08390,54.60600,17,3,2,...,-1,0,1,1,2,E01032560,3,1,0.0,1.0
3,202417S104724,2024,17S104724,440460.0,517548.0,-1.37599,54.55157,17,3,2,...,-1,0,2,1,1,E01012269,3,1,0.0,1.0
4,202517S106525,2025,17S106525,444331.0,518776.0,-1.31596,54.56228,17,2,2,...,-1,0,1,1,2,E01035199,7,1,1.0,0.0


In [54]:
collisions.shape    # check the dimensions
collisions.info()   # column names and data types
collisions.columns.tolist()
collisions["collision_year"].value_counts().sort_index()


<class 'pandas.DataFrame'>
RangeIndex: 513801 entries, 0 to 513800
Data columns (total 44 columns):
 #   Column                                            Non-Null Count   Dtype  
---  ------                                            --------------   -----  
 0   collision_index                                   513801 non-null  str    
 1   collision_year                                    513801 non-null  int64  
 2   collision_ref_no                                  513801 non-null  str    
 3   location_easting_osgr                             513748 non-null  float64
 4   location_northing_osgr                            513748 non-null  float64
 5   longitude                                         513748 non-null  float64
 6   latitude                                          513748 non-null  float64
 7   police_force                                      513801 non-null  int64  
 8   collision_severity                                513801 non-null  int64  
 9   number_of_vehic

collision_year
2021    101087
2022    106004
2023    104258
2024    100927
2025    101525
Name: count, dtype: int64

In [55]:
# count missing values in each column
collisions.isna().sum().sort_values(ascending=False)

location_easting_osgr                               53
location_northing_osgr                              53
longitude                                           53
latitude                                            53
collision_index                                      0
carriageway_hazards_historic                         0
second_road_number                                   0
pedestrian_crossing_human_control_historic           0
pedestrian_crossing_physical_facilities_historic     0
pedestrian_crossing                                  0
light_conditions                                     0
weather_conditions                                   0
road_surface_conditions                              0
special_conditions_at_site                           0
urban_or_rural_area                                  0
carriageway_hazards                                  0
junction_control                                     0
did_police_officer_attend_scene_of_accident          0
trunk_road

In [56]:
important_columns = [
    "collision_index",
    "collision_year",
    "date",
    "time",
    "location_easting_osgr",
    "location_northing_osgr",
    "local_authority_ons_district"
]

collisions[important_columns].isna().sum()

collision_index                  0
collision_year                   0
date                             0
time                             0
location_easting_osgr           53
location_northing_osgr          53
local_authority_ons_district     0
dtype: int64

In [57]:
#count duplicates
collisions.duplicated(subset=["collision_index"]).sum()

np.int64(0)

In [58]:
# filter for London boroughs (E09 codes)
# with .copy() explicit independent copy is created
london_collisions = collisions[collisions["local_authority_ons_district"].str.startswith("E09", na=False)].copy()

In [59]:
# standardize column names

london_collisions = london_collisions.rename(columns={
    "location_easting_osgr": "easting",
    "location_northing_osgr": "northing",
    "local_authority_ons_district": "borough_code"
})

In [60]:
london_collisions[[
    "collision_index",
    "easting",
    "northing",
    "borough_code"
]].head()

,collision_index,easting,northing,borough_code
267,2024481507940,532113.0,181737.0,E09000001
268,2025010590101,548177.0,188052.0,E09000002
269,2025010561206,524331.0,188977.0,E09000003
270,2025010577340,526851.0,191831.0,E09000003
271,2025010600027,521286.0,189001.0,E09000003


In [61]:
print("Original dataset:", collisions.shape)
print("London dataset:", london_collisions.shape)

Original dataset: (513801, 44)
London dataset: (111335, 44)


In [65]:
# verify that all borough codes start with E09
london_collisions["borough_code"].unique()

<StringArray>
['E09000001', 'E09000002', 'E09000003', 'E09000004', 'E09000005', 'E09000006',
 'E09000007', 'E09000008', 'E09000009', 'E09000010', 'E09000011', 'E09000012',
 'E09000013', 'E09000014', 'E09000015', 'E09000016', 'E09000017', 'E09000018',
 'E09000019', 'E09000020', 'E09000021', 'E09000022', 'E09000023', 'E09000024',
 'E09000025', 'E09000026', 'E09000028', 'E09000029', 'E09000030', 'E09000031',
 'E09000032', 'E09000033', 'E09000027']
Length: 33, dtype: str

In [63]:

# check number of rows removed
print(f"Rows removed: {len(collisions) - len(london_collisions)}")

Rows removed: 402466


In [64]:
london_collisions.head()

,collision_index,collision_year,collision_ref_no,easting,northing,longitude,latitude,police_force,collision_severity,number_of_vehicles,...,carriageway_hazards_historic,carriageway_hazards,urban_or_rural_area,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight
267,2024481507940,2024,481507940,532113.0,181737.0,-0.09728,51.51911,48,3,2,...,-1,0,1,2,2,E01000001,3,1,0.0,1.0
268,2025010590101,2025,010590101,548177.0,188052.0,0.13674,51.57187,1,3,2,...,-1,0,1,1,2,E01000112,3,1,0.0,1.0
269,2025010561206,2025,010561206,524331.0,188977.0,-0.20682,51.58594,1,2,1,...,-1,0,1,1,2,E01000225,6,1,1.0,0.0
270,2025010577340,2025,010577340,526851.0,191831.0,-0.16944,51.61103,1,2,2,...,-1,0,1,1,2,E01000318,5,1,1.0,0.0
271,2025010600027,2025,010600027,521286.0,189001.0,-0.25074,51.58682,1,3,2,...,-1,0,1,1,2,E01033925,3,1,0.0,1.0


In [67]:
london_collisions.isin([-1]).sum()

collision_index                                          0
collision_year                                           0
collision_ref_no                                         0
easting                                                  0
northing                                                 0
longitude                                                0
latitude                                                 0
police_force                                             0
collision_severity                                       0
number_of_vehicles                                       0
number_of_casualties                                     0
date                                                     0
day_of_week                                              0
time                                                     0
local_authority_district                            111335
borough_code                                             0
local_authority_highway                                 